In [28]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from tqdm.auto import tqdm
import re

In [29]:
comments = pd.read_csv(r'C:\Users\New Owner\OneDrive\Documents\April DS Code Jam\youtube_sentiment_analysis\datasets\archive\UScomments.csv', on_bad_lines='skip', encoding= 'utf-8', low_memory=False) # Load the dataset

In [30]:
comments.head(10)

,video_id,comment_text,likes,replies
0,XpVt6Z1Gjjo,Logan Paul it's yo big day ‼️‼️‼️,4,0
1,XpVt6Z1Gjjo,I've been following you from the start of your...,3,0
2,XpVt6Z1Gjjo,Say hi to Kong and maverick for me,3,0
3,XpVt6Z1Gjjo,MY FAN . attendance,3,0
4,XpVt6Z1Gjjo,trending 😉,3,0
5,XpVt6Z1Gjjo,#1 on trending AYYEEEEE,3,0
6,XpVt6Z1Gjjo,The end though 😭👍🏻❤️,4,0
7,XpVt6Z1Gjjo,#1 trending!!!!!!!!!,3,0
8,XpVt6Z1Gjjo,Happy one year vlogaversary,3,0
9,XpVt6Z1Gjjo,You and your shit brother may have single hand...,0,0


In [31]:
videos = pd.read_csv(r'C:\Users\New Owner\OneDrive\Documents\April DS Code Jam\youtube_sentiment_analysis\datasets\archive\USvideos.csv', on_bad_lines='skip', encoding= 'utf-8').head(10) # Display the first 10 rows of the dataset

In [32]:
videos.head(10)

,video_id,title,channel_title,category_id,tags,views,likes,dislikes,comment_total,thumbnail_link,date
0,XpVt6Z1Gjjo,1 YEAR OF VLOGGING -- HOW LOGAN PAUL CHANGED Y...,Logan Paul Vlogs,24,logan paul vlog|logan paul|logan|paul|olympics...,4394029,320053,5931,46245,https://i.ytimg.com/vi/XpVt6Z1Gjjo/default.jpg,13.09
1,K4wEI5zhHB0,iPhone X — Introducing iPhone X — Apple,Apple,28,Apple|iPhone 10|iPhone Ten|iPhone|Portrait Lig...,7860119,185853,26679,0,https://i.ytimg.com/vi/K4wEI5zhHB0/default.jpg,13.09
2,cLdxuaxaQwc,My Response,PewDiePie,22,[none],5845909,576597,39774,170708,https://i.ytimg.com/vi/cLdxuaxaQwc/default.jpg,13.09
3,WYYvHb03Eog,Apple iPhone X first look,The Verge,28,apple iphone x hands on|Apple iPhone X|iPhone ...,2642103,24975,4542,12829,https://i.ytimg.com/vi/WYYvHb03Eog/default.jpg,13.09
4,sjlHnJvXdQs,iPhone X (parody),jacksfilms,23,jacksfilms|parody|parodies|iphone|iphone x|iph...,1168130,96666,568,6666,https://i.ytimg.com/vi/sjlHnJvXdQs/default.jpg,13.09
5,cMKX2tE5Luk,The Disaster Artist | Official Trailer HD | A24,A24,1,a24|a24 films|a24 trailers|independent films|t...,1311445,34507,544,3040,https://i.ytimg.com/vi/cMKX2tE5Luk/default.jpg,13.09
6,8wNr-NQImFg,"The Check In: HUD, Ben Carson and Hurricanes",Late Night with Seth Meyers,23,Late night|Seth Meyers|check in|hud|Ben Carson...,666169,9985,297,1071,https://i.ytimg.com/vi/8wNr-NQImFg/default.jpg,13.09
7,_HTXMhKWqnA,iPhone X Impressions & Hands On!,Marques Brownlee,28,iPhone X|iphone x|iphone 10|iPhone X impressio...,1728614,74062,2180,15297,https://i.ytimg.com/vi/_HTXMhKWqnA/default.jpg,13.09
8,_ANP3HR1jsM,ATTACKED BY A POLICE DOG!!,RomanAtwoodVlogs,22,Roman Atwood|Roman|Atwood|roman atwood vlogs|f...,1338533,69687,678,5643,https://i.ytimg.com/vi/_ANP3HR1jsM/default.jpg,13.09
9,zgLtEob6X-Q,Honest Trailers - The Mummy (2017),Screen Junkies,1,screenjunkies|screen junkies|screenjunkies new...,1056891,29943,878,4046,https://i.ytimg.com/vi/zgLtEob6X-Q/default.jpg,13.09


## Calculating Like-to-Dislike Ratio

In [33]:
videos['l_d_ratio'] = videos['likes'] / (videos['likes'] + videos['dislikes']) # Calculate the like-dislike ratio
videos = videos.drop(columns=['date']) # Drop the date column
videos.head(10)

,video_id,title,channel_title,category_id,tags,views,likes,dislikes,comment_total,thumbnail_link,l_d_ratio
0,XpVt6Z1Gjjo,1 YEAR OF VLOGGING -- HOW LOGAN PAUL CHANGED Y...,Logan Paul Vlogs,24,logan paul vlog|logan paul|logan|paul|olympics...,4394029,320053,5931,46245,https://i.ytimg.com/vi/XpVt6Z1Gjjo/default.jpg,0.981806
1,K4wEI5zhHB0,iPhone X — Introducing iPhone X — Apple,Apple,28,Apple|iPhone 10|iPhone Ten|iPhone|Portrait Lig...,7860119,185853,26679,0,https://i.ytimg.com/vi/K4wEI5zhHB0/default.jpg,0.874471
2,cLdxuaxaQwc,My Response,PewDiePie,22,[none],5845909,576597,39774,170708,https://i.ytimg.com/vi/cLdxuaxaQwc/default.jpg,0.935471
3,WYYvHb03Eog,Apple iPhone X first look,The Verge,28,apple iphone x hands on|Apple iPhone X|iPhone ...,2642103,24975,4542,12829,https://i.ytimg.com/vi/WYYvHb03Eog/default.jpg,0.846123
4,sjlHnJvXdQs,iPhone X (parody),jacksfilms,23,jacksfilms|parody|parodies|iphone|iphone x|iph...,1168130,96666,568,6666,https://i.ytimg.com/vi/sjlHnJvXdQs/default.jpg,0.994158
5,cMKX2tE5Luk,The Disaster Artist | Official Trailer HD | A24,A24,1,a24|a24 films|a24 trailers|independent films|t...,1311445,34507,544,3040,https://i.ytimg.com/vi/cMKX2tE5Luk/default.jpg,0.984480
6,8wNr-NQImFg,"The Check In: HUD, Ben Carson and Hurricanes",Late Night with Seth Meyers,23,Late night|Seth Meyers|check in|hud|Ben Carson...,666169,9985,297,1071,https://i.ytimg.com/vi/8wNr-NQImFg/default.jpg,0.971115
7,_HTXMhKWqnA,iPhone X Impressions & Hands On!,Marques Brownlee,28,iPhone X|iphone x|iphone 10|iPhone X impressio...,1728614,74062,2180,15297,https://i.ytimg.com/vi/_HTXMhKWqnA/default.jpg,0.971407
8,_ANP3HR1jsM,ATTACKED BY A POLICE DOG!!,RomanAtwoodVlogs,22,Roman Atwood|Roman|Atwood|roman atwood vlogs|f...,1338533,69687,678,5643,https://i.ytimg.com/vi/_ANP3HR1jsM/default.jpg,0.990365
9,zgLtEob6X-Q,Honest Trailers - The Mummy (2017),Screen Junkies,1,screenjunkies|screen junkies|screenjunkies new...,1056891,29943,878,4046,https://i.ytimg.com/vi/zgLtEob6X-Q/default.jpg,0.971513


## Text Cleaning

In [34]:
def clear_text(text):
    text = text.lower() # Convert to lowercase
    pattern = r'[^a-zA-Z\s]' # Regular expression pattern to match special characters and punctuation 
    text = re.sub(pattern, " ", text) # Remove special characters, including punctuation
    return text

In [35]:
comments['comment_text'] = comments['comment_text'].astype(str).apply(clear_text)
comments['comment_text'].head(10)

0                    logan paul it s yo big day       
1    i ve been following you from the start of your...
2                   say hi to kong and maverick for me
3                                  my fan   attendance
4                                           trending  
5                                 on trending ayyeeeee
6                                 the end though      
7                                    trending         
8                          happy one year vlogaversary
9    you and your shit brother may have single hand...
Name: comment_text, dtype: object

## Tokenization

In [36]:
nltk.download('punkt_tab')

comments['tokenized_text'] = comments['comment_text'].fillna("").astype(str).apply(word_tokenize) # Apply tokenization to the 'comment_text' column



[nltk_data] Downloading package punkt_tab to C:\Users\New
[nltk_data]     Owner\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [37]:
comments['tokenized_text'].head(10)

0                   [logan, paul, it, s, yo, big, day]
1    [i, ve, been, following, you, from, the, start...
2          [say, hi, to, kong, and, maverick, for, me]
3                                [my, fan, attendance]
4                                           [trending]
5                             [on, trending, ayyeeeee]
6                                   [the, end, though]
7                                           [trending]
8                     [happy, one, year, vlogaversary]
9    [you, and, your, shit, brother, may, have, sin...
Name: tokenized_text, dtype: object

## Stop Words Removal

In [38]:
nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

# Remove stop words from the tokenized text
comments['tokenized_text'] = comments['tokenized_text'].apply(
    lambda tokens: [word for word in tokens if word not in stop_words]
)

[nltk_data] Downloading package stopwords to C:\Users\New
[nltk_data]     Owner\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [39]:
comments['tokenized_text'].head(20)

0                           [logan, paul, yo, big, day]
1        [following, start, vine, channel, seen, vlogs]
2                             [say, hi, kong, maverick]
3                                     [fan, attendance]
4                                            [trending]
5                                  [trending, ayyeeeee]
6                                         [end, though]
7                                            [trending]
8                      [happy, one, year, vlogaversary]
9     [shit, brother, may, single, handedly, ruined,...
10                                  [mini, logan, paul]
11    [dear, logan, really, wan, na, get, merch, mon...
12    [honestly, evan, annoying, like, funny, watchi...
13                        [casey, still, better, logan]
14                 [aw, geez, rick, guy, face, youtube]
15                                [happy, cause, movie]
16    [ayyyyoooo, logang, hard, vlog, watch, logan, ...
17      [bro, didnt, u, give, merch, johannes, u

## Lemmetization

In [40]:
nltk.download('wordnet')

lemmatizer = WordNetLemmatizer()

# Apply lemmatization to each tokenized comment
comments['tokenized_text'] = comments['tokenized_text'].apply(
    lambda tokens: [lemmatizer.lemmatize(word) for word in tokens])

[nltk_data] Downloading package wordnet to C:\Users\New
[nltk_data]     Owner\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [41]:
comments['tokenized_text'].head(20)

0                           [logan, paul, yo, big, day]
1        [following, start, vine, channel, seen, vlogs]
2                             [say, hi, kong, maverick]
3                                     [fan, attendance]
4                                            [trending]
5                                  [trending, ayyeeeee]
6                                         [end, though]
7                                            [trending]
8                      [happy, one, year, vlogaversary]
9     [shit, brother, may, single, handedly, ruined,...
10                                  [mini, logan, paul]
11    [dear, logan, really, wan, na, get, merch, mon...
12    [honestly, evan, annoying, like, funny, watchi...
13                        [casey, still, better, logan]
14                 [aw, geez, rick, guy, face, youtube]
15                                [happy, cause, movie]
16    [ayyyyoooo, logang, hard, vlog, watch, logan, ...
17      [bro, didnt, u, give, merch, johannes, u

In [42]:
# Join the tokenized words back into a single string and save it back to the column
comments['tokenized_text'] = comments['tokenized_text'].apply(lambda tokens: ' '.join(tokens))
comments = comments.drop(columns=['comment_text']) # Drop the original 'comment_text' column

In [43]:
comments['tokenized_text'].head(10)

0                                logan paul yo big day
1              following start vine channel seen vlogs
2                                 say hi kong maverick
3                                       fan attendance
4                                             trending
5                                    trending ayyeeeee
6                                           end though
7                                             trending
8                          happy one year vlogaversary
9    shit brother may single handedly ruined youtub...
Name: tokenized_text, dtype: object

In [44]:
comments.head(10)

,video_id,likes,replies,tokenized_text
0,XpVt6Z1Gjjo,4,0,logan paul yo big day
1,XpVt6Z1Gjjo,3,0,following start vine channel seen vlogs
2,XpVt6Z1Gjjo,3,0,say hi kong maverick
3,XpVt6Z1Gjjo,3,0,fan attendance
4,XpVt6Z1Gjjo,3,0,trending
5,XpVt6Z1Gjjo,3,0,trending ayyeeeee
6,XpVt6Z1Gjjo,4,0,end though
7,XpVt6Z1Gjjo,3,0,trending
8,XpVt6Z1Gjjo,3,0,happy one year vlogaversary
9,XpVt6Z1Gjjo,0,0,shit brother may single handedly ruined youtub...


In [45]:
videos.head(10)

,video_id,title,channel_title,category_id,tags,views,likes,dislikes,comment_total,thumbnail_link,l_d_ratio
0,XpVt6Z1Gjjo,1 YEAR OF VLOGGING -- HOW LOGAN PAUL CHANGED Y...,Logan Paul Vlogs,24,logan paul vlog|logan paul|logan|paul|olympics...,4394029,320053,5931,46245,https://i.ytimg.com/vi/XpVt6Z1Gjjo/default.jpg,0.981806
1,K4wEI5zhHB0,iPhone X — Introducing iPhone X — Apple,Apple,28,Apple|iPhone 10|iPhone Ten|iPhone|Portrait Lig...,7860119,185853,26679,0,https://i.ytimg.com/vi/K4wEI5zhHB0/default.jpg,0.874471
2,cLdxuaxaQwc,My Response,PewDiePie,22,[none],5845909,576597,39774,170708,https://i.ytimg.com/vi/cLdxuaxaQwc/default.jpg,0.935471
3,WYYvHb03Eog,Apple iPhone X first look,The Verge,28,apple iphone x hands on|Apple iPhone X|iPhone ...,2642103,24975,4542,12829,https://i.ytimg.com/vi/WYYvHb03Eog/default.jpg,0.846123
4,sjlHnJvXdQs,iPhone X (parody),jacksfilms,23,jacksfilms|parody|parodies|iphone|iphone x|iph...,1168130,96666,568,6666,https://i.ytimg.com/vi/sjlHnJvXdQs/default.jpg,0.994158
5,cMKX2tE5Luk,The Disaster Artist | Official Trailer HD | A24,A24,1,a24|a24 films|a24 trailers|independent films|t...,1311445,34507,544,3040,https://i.ytimg.com/vi/cMKX2tE5Luk/default.jpg,0.984480
6,8wNr-NQImFg,"The Check In: HUD, Ben Carson and Hurricanes",Late Night with Seth Meyers,23,Late night|Seth Meyers|check in|hud|Ben Carson...,666169,9985,297,1071,https://i.ytimg.com/vi/8wNr-NQImFg/default.jpg,0.971115
7,_HTXMhKWqnA,iPhone X Impressions & Hands On!,Marques Brownlee,28,iPhone X|iphone x|iphone 10|iPhone X impressio...,1728614,74062,2180,15297,https://i.ytimg.com/vi/_HTXMhKWqnA/default.jpg,0.971407
8,_ANP3HR1jsM,ATTACKED BY A POLICE DOG!!,RomanAtwoodVlogs,22,Roman Atwood|Roman|Atwood|roman atwood vlogs|f...,1338533,69687,678,5643,https://i.ytimg.com/vi/_ANP3HR1jsM/default.jpg,0.990365
9,zgLtEob6X-Q,Honest Trailers - The Mummy (2017),Screen Junkies,1,screenjunkies|screen junkies|screenjunkies new...,1056891,29943,878,4046,https://i.ytimg.com/vi/zgLtEob6X-Q/default.jpg,0.971513


In [46]:
comments.head(10)

,video_id,likes,replies,tokenized_text
0,XpVt6Z1Gjjo,4,0,logan paul yo big day
1,XpVt6Z1Gjjo,3,0,following start vine channel seen vlogs
2,XpVt6Z1Gjjo,3,0,say hi kong maverick
3,XpVt6Z1Gjjo,3,0,fan attendance
4,XpVt6Z1Gjjo,3,0,trending
5,XpVt6Z1Gjjo,3,0,trending ayyeeeee
6,XpVt6Z1Gjjo,4,0,end though
7,XpVt6Z1Gjjo,3,0,trending
8,XpVt6Z1Gjjo,3,0,happy one year vlogaversary
9,XpVt6Z1Gjjo,0,0,shit brother may single handedly ruined youtub...


## Creating Positive to Negative Comments Ratio

In [ ]:
# 1.Calculate the polarity score of an individual comment
from textblob import TextBlob

# Function to calculate sentiment polarity
def get_sentiment(text):
    analysis = TextBlob(text)
    return analysis.sentiment.polarity

# Apply the function to the 'tokenized_text' column
comments['polarity']= comments['tokenized_text'].apply(get_sentiment)

comments.head(10)

,video_id,likes,replies,tokenized_text,polarity
0,XpVt6Z1Gjjo,4,0,logan paul yo big day,0.00000
1,XpVt6Z1Gjjo,3,0,following start vine channel seen vlogs,0.00000
2,XpVt6Z1Gjjo,3,0,say hi kong maverick,0.00000
3,XpVt6Z1Gjjo,3,0,fan attendance,0.00000
4,XpVt6Z1Gjjo,3,0,trending,0.00000
5,XpVt6Z1Gjjo,3,0,trending ayyeeeee,0.00000
6,XpVt6Z1Gjjo,4,0,end though,0.00000
7,XpVt6Z1Gjjo,3,0,trending,0.00000
8,XpVt6Z1Gjjo,3,0,happy one year vlogaversary,0.80000
9,XpVt6Z1Gjjo,0,0,shit brother may single handedly ruined youtub...,-0.02381


In [ ]:
# 2.Classify the sentiment based on polarity
comments['sentiment_bin'] = comments['polarity'].apply(lambda x: 'positive' if x > 0 else ('negative' if x < 0 else 'neutral'))
sentiment_share = comments.groupby('video_id')['sentiment_bin'].value_counts().unstack().fillna(0).head(20) # Display the sentiment counts for each video

In [ ]:
# 3.Calculate Positive-to-Negative Ratio
sentiment_share['ratio'] = sentiment_share['positive'] / sentiment_share['negative']
sentiment_share.fillna(0, inplace=True)  # Handle cases where there are no negative comments


In [ ]:
# 4.Calculate Percentage of Neutral Comments used in the analysis
# neutral percentage is used to help calculate the emoti0onal charge related to audience reaction.
sentiment_share['neutral_percentage'] = (sentiment_share['neutral'] / (sentiment_share['positive'] + sentiment_share['negative'] + sentiment_share['neutral'])) * 100

In [55]:
sentiment_share.head(10)

sentiment_bin,negative,neutral,positive,ratio,neutral_percentage
video_id,,,,,
--JinobXWPk,18.0,52.0,30.0,1.666667,52.000000
-1fzGnFwz9M,19.0,28.0,53.0,2.789474,28.000000
-3AGlBYyLjo,2.0,2.0,0.0,0.000000,50.000000
-5sCWsLlTCI,17.0,22.0,28.0,1.647059,32.835821
-6Zc8Co2H3w,34.0,173.0,193.0,5.676471,43.250000
-AJyaVduxCc,44.0,116.0,131.0,2.977273,39.862543
-B9z3az6Axc,55.0,207.0,238.0,4.327273,41.400000
-C-LJUD2LWU,36.0,49.0,115.0,3.194444,24.500000
-CEuQhqNzz4,8.0,161.0,65.0,8.125000,68.803419


In [56]:
videos.head(10)

,video_id,title,channel_title,category_id,tags,views,likes,dislikes,comment_total,thumbnail_link,l_d_ratio
0,XpVt6Z1Gjjo,1 YEAR OF VLOGGING -- HOW LOGAN PAUL CHANGED Y...,Logan Paul Vlogs,24,logan paul vlog|logan paul|logan|paul|olympics...,4394029,320053,5931,46245,https://i.ytimg.com/vi/XpVt6Z1Gjjo/default.jpg,0.981806
1,K4wEI5zhHB0,iPhone X — Introducing iPhone X — Apple,Apple,28,Apple|iPhone 10|iPhone Ten|iPhone|Portrait Lig...,7860119,185853,26679,0,https://i.ytimg.com/vi/K4wEI5zhHB0/default.jpg,0.874471
2,cLdxuaxaQwc,My Response,PewDiePie,22,[none],5845909,576597,39774,170708,https://i.ytimg.com/vi/cLdxuaxaQwc/default.jpg,0.935471
3,WYYvHb03Eog,Apple iPhone X first look,The Verge,28,apple iphone x hands on|Apple iPhone X|iPhone ...,2642103,24975,4542,12829,https://i.ytimg.com/vi/WYYvHb03Eog/default.jpg,0.846123
4,sjlHnJvXdQs,iPhone X (parody),jacksfilms,23,jacksfilms|parody|parodies|iphone|iphone x|iph...,1168130,96666,568,6666,https://i.ytimg.com/vi/sjlHnJvXdQs/default.jpg,0.994158
5,cMKX2tE5Luk,The Disaster Artist | Official Trailer HD | A24,A24,1,a24|a24 films|a24 trailers|independent films|t...,1311445,34507,544,3040,https://i.ytimg.com/vi/cMKX2tE5Luk/default.jpg,0.984480
6,8wNr-NQImFg,"The Check In: HUD, Ben Carson and Hurricanes",Late Night with Seth Meyers,23,Late night|Seth Meyers|check in|hud|Ben Carson...,666169,9985,297,1071,https://i.ytimg.com/vi/8wNr-NQImFg/default.jpg,0.971115
7,_HTXMhKWqnA,iPhone X Impressions & Hands On!,Marques Brownlee,28,iPhone X|iphone x|iphone 10|iPhone X impressio...,1728614,74062,2180,15297,https://i.ytimg.com/vi/_HTXMhKWqnA/default.jpg,0.971407
8,_ANP3HR1jsM,ATTACKED BY A POLICE DOG!!,RomanAtwoodVlogs,22,Roman Atwood|Roman|Atwood|roman atwood vlogs|f...,1338533,69687,678,5643,https://i.ytimg.com/vi/_ANP3HR1jsM/default.jpg,0.990365
9,zgLtEob6X-Q,Honest Trailers - The Mummy (2017),Screen Junkies,1,screenjunkies|screen junkies|screenjunkies new...,1056891,29943,878,4046,https://i.ytimg.com/vi/zgLtEob6X-Q/default.jpg,0.971513


In [ ]:
#videos_mass = videos.merge(sent_prop[['comment_sent_ratio']], left_on='video_id', right_index=True, how='left') # Merge the sentiment ratio with the videos DataFrame

NameError: name 'sent_prop' is not defined

In [ ]:
sent_prop.head(10)

sentiment_class,negative,positive,comment_sent_ratio
video_id,,,
--JinobXWPk,70.0,30.0,0.300000
-1fzGnFwz9M,47.0,53.0,0.530000
-3AGlBYyLjo,4.0,0.0,0.000000
-5sCWsLlTCI,39.0,28.0,0.417910
-6Zc8Co2H3w,207.0,193.0,0.482500
-AJyaVduxCc,160.0,131.0,0.450172
-B9z3az6Axc,262.0,238.0,0.476000
-C-LJUD2LWU,85.0,115.0,0.575000
-CEuQhqNzz4,169.0,65.0,0.277778
